# Продвинутый PySpark: Агрегации и Оконные Функции
**Участники:**
* **Ричард Фейнман (PhD Teacher):** Объясняет сложные вещи на пальцах, используя аналогии из жизни.
* **Senior Big Data Analyst:** Отвечает за продакшен-код, оптимизацию, архитектуру и "подводные камни".
* **ML Researcher:** Смотрит на данные как на признаки (фичи) для моделей машинного обучения.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Инициализация сессии
spark = SparkSession.builder.appName("AdvancedPySpark").getOrCreate()

# Создаем тестовый датафрейм
data = [
    ("Alice", "Sales", 3000, "2023-01-01"),
    ("Bob", "Sales", 4000, "2023-01-02"),
    ("Charlie", "Sales", 3500, "2023-01-03"),
    ("Alice", "Sales", 5000, "2023-02-01"),
    ("Eve", "IT", 7000, "2023-01-01"),
    ("Frank", "IT", 8000, "2023-01-02"),
    ("Eve", "IT", 7500, "2023-02-01"),
]
df = spark.createDataFrame(data, ["name", "department", "salary", "date"])
df.show()

## 1. Select и Filter (Базовые операции)
**Ричард Фейнман:** Представьте, что датафрейм — это огромная толпа людей, пришедших на концерт. `select` и `filter` — это просто фейсконтроль на входе. Мы выбираем только тех, кто, например, купил VIP-билет (`filter`), и спрашиваем у них только имя и ряд (`select`), игнорируя остальную информацию. Это очень простая отбраковка, давайте быстро на нее посмотрим.

**Senior Big Data Analyst:** В PySpark используем `select` для выбора нужных колонок и `filter` (или синоним `where`) для фильтрации строк. Звучит просто, но есть золотое правило оптимизации: **всегда фильтруйте данные как можно раньше**. Механизм *Predicate Pushdown* отсеет ненужные данные еще на этапе чтения с диска (например, из Parquet), что сэкономит кучу CPU, памяти и Network I/O.

In [ ]:
# Быстрый select и filter
filtered_df = df.select("name", "department", "salary").filter(F.col("salary") > 4000)
filtered_df.show()

## 2. Агрегации и GroupBy
**Ричард Фейнман:** Теперь представьте, что мы хотим узнать, сколько всего денег зарабатывает каждый отдел в компании. Нам нужно собрать всех людей в кучки по табличкам на дверях кабинетов (это `groupBy`), а потом внутри каждой кучки сложить их зарплаты (это агрегация `sum`). Агрегация — это процесс превращения множества мелких индивидуальных деталей в один крупный, обобщенный факт.

**Senior Big Data Analyst:** В распределенных системах агрегации работают через механизм **Shuffle** — данные физически пересылаются по сети между серверами-экзекьюторами, чтобы строки с одинаковым ключом оказались на одном узле. Это тяжелая операция, избегайте ее без нужды.
Какие бывают методы агрегации:
* `F.sum()`, `F.avg()`, `F.min()`, `F.max()` — классические математические операции.
* `F.count()`, `F.countDistinct()` — подсчет. Внимание: `countDistinct` требует полного шаффла всех уникальных значений. Если вам не нужна 100% точность (например, уникальных пользователей миллионы), используйте `F.approx_count_distinct()` — он работает в разы быстрее через HyperLogLog.
* `F.collect_list()`, `F.collect_set()` — собирают все значения группы в единый массив (с дубликатами или без). Осторожно: может вызвать `OutOfMemoryError`, если ключей мало, а значений много (эффект data skew).
* `F.first()`, `F.last()` — берет первое/последнее значение. Обязательно требуют предварительной сортировки внутри группы, иначе результат будет случайным из-за распределенной природы Spark!

**ML Researcher:** Для ML агрегации — это основной источник генерации признаков (feature engineering). Например, вычисление средней суммы покупок пользователя за месяц (`avg`), максимума трат (`max`). А функция `collect_list` безумно удобна для создания sequence-фичей, которые потом скармливаются нейросетям (RNN/Transformers) для предсказания следующего действия!

In [ ]:
# Применяем GroupBy и несколько агрегаций
agg_df = df.groupBy("department").agg(
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.max("salary").alias("max_salary"),
    F.countDistinct("name").alias("unique_employees"),
    F.collect_list("name").alias("all_employees_list")
)
agg_df.show(truncate=False)

## 3. Оконные функции (Window Functions)

**Ричард Фейнман:** А что, если мы хотим знать среднюю зарплату по отделу, но при этом *не хотим* сжимать наших людей в одну строку для каждого отдела? Мы хотим, чтобы каждый человек остался в таблице как самостоятельная единица, но рядом с ним появилась колонка "средняя зарплата его отдела". 
Оконная функция — это как рамка окна, через которую каждый человек смотрит на свою группу. Он видит всех своих коллег (`partitionBy`), может сравнить себя с ними, но при этом не сливается с ними в одну массу!

**Senior Big Data Analyst:** Оконки мощнее и гибче обычного `groupBy`. Для их использования мы определяем спецификацию окна `WindowSpec`.
Из чего состоит окно и какие методы применяются:
* `Window.partitionBy("col")` — делит данные на независимые группы (окна). Аналог GROUP BY, но не уменьшает количество строк.
* `Window.orderBy("col")` — задает порядок обработки строк *внутри* окна. Крайне важно для функций вроде `lag` (предыдущее значение), `lead` (следующее), `row_number` (номер строки), или когда мы считаем нарастающий итог.
* `Window.rowsBetween(start, end)` и `Window.rangeBetween(start, end)` — определяют скользящие границы фрейма относительно текущей строки. Например, `Window.unboundedPreceding` (от самого начала окна) до `Window.currentRow` (текущая строка). `rowsBetween` считает физические строки, а `rangeBetween` логические значения.

Сам метод `.over(windowSpec)` — это пусковой крючок. Он применяется к функции колонки и говорит Spark'у: *"Посчитай эту агрегацию или функцию, используя вот это конкретное окно"*.

**ML Researcher:** Оконные функции — это чистое золото для Time-Series фичей. Нарастающий итог (cumulative sum), разница с предыдущим днем (diff), сдвиги во времени (lag) — всё это делается элегантно через окна. Алгоритмы вроде градиентного бустинга (XGBoost/CatBoost) обожают фичи, описывающие динамику изменений во времени для конкретного пользователя или объекта!

In [ ]:
# Определяем разные спецификации окон

# Окно 1: просто группа по отделу (подойдет для агрегаций без сжатия)
window_dept = Window.partitionBy("department")

# Окно 2: группа по отделу + сортировка по дате (нужно для ранжирования, лагов и нарастающих итогов)
window_dept_time = Window.partitionBy("department").orderBy("date")

# Применяем оконные функции через метод .over()
window_df = df \
    .withColumn("dept_avg_salary", F.avg("salary").over(window_dept)) \
    .withColumn("salary_rank_in_dept", F.rank().over(window_dept_time)) \
    .withColumn("prev_day_salary", F.lag("salary", 1).over(window_dept_time)) \
    .withColumn("diff_with_prev", F.col("salary") - F.col("prev_day_salary")) \
    .withColumn("running_total_salary", F.sum("salary").over(window_dept_time))

window_df.orderBy("department", "date").show(truncate=False)

## Итоги и Советы
**Senior Big Data Analyst:** Запомните коварную вещь: использование `orderBy` в окне **автоматически** добавляет скрытый фрейм `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. То есть функция будет считаться нарастающим итогом до текущей строки! Если вам нужна была просто сумма по всему `partitionBy`, но вы зачем-то добавили `orderBy` — вы получите баг (нарастающую сумму вместо общей).

**Ричард Фейнман:** Физика оконных функций проста: вы стоите в своей строке, открываете окно нужного размера (`rowsBetween`), смотрите на своих соседей по группе (`partitionBy`), выстраиваете их по росту или дате (`orderBy`) и считаете среднее, или просто просите соседа слева дать списать (`lag`).

**ML Researcher:** Удачи с генерацией фичей! Правильно собранные агрегации и темпоральные сдвиги из оконных функций легко поднимут качество вашей модели (ROC-AUC) на пару пунктов. Экспериментируйте с размерами окон (за 7 дней, 30 дней) — в них кроется много инсайтов!